# 00 · Data preparation

Streams **FineWeb-Edu (sample-10BT)**, tokenises it with the Llama-2 SentencePiece tokenizer (32k vocab) and writes `train.bin` (51M tokens) + `val.bin` (1M tokens) as uint16 to Google Drive. Run once; notebooks 01–03 reuse the files.

In [ ]:
# --- Colab setup: GPU runtime (Runtime > Change runtime type > T4 GPU) ---
REPO_URL = "https://github.com/gaurkhare/gaurav-eagv5-s13.git"
import os, sys, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era_a13"          # data + results persist across notebooks
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    os.chdir("/content/repo")
else:
    WORK = os.path.abspath("..")                      # running locally from notebooks/
    os.chdir(WORK)
sys.path.insert(0, os.getcwd())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA_DIR, RESULTS = f"{WORK}/data", f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no nvidia GPU"

In [ ]:
!pip -q install datasets transformers

In [ ]:
from revllm.data import prepare
if not os.path.exists(f"{DATA_DIR}/train.bin"):
    vocab = prepare(DATA_DIR, train_tokens=51_000_000, val_tokens=1_000_000)
    print("vocab", vocab)
!ls -lh {DATA_DIR}

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from revllm.data import TOKENIZER
tok = AutoTokenizer.from_pretrained(TOKENIZER)
arr = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r")
print(len(arr), "train tokens; max id", arr[:5_000_000].max())
print(tok.decode(arr[:200].tolist()))